In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import root_mean_squared_error

In [2]:
DATA_PATH = "m5-forecasting-accuracy"

calendar = pd.read_csv(f"{DATA_PATH}/calendar.csv")
prices = pd.read_csv(f"{DATA_PATH}/sell_prices.csv")
sales = pd.read_csv(f"{DATA_PATH}/sales_train_validation.csv")

In [3]:
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
value_cols = [col for col in sales.columns if col.startswith("d_")]

df = sales.melt(id_vars=id_cols, value_vars=value_cols,
                var_name="d", value_name="sales")

In [4]:
df = df.merge(calendar, on="d", how="left")


In [5]:
df = df.merge(prices,
              on=["store_id", "item_id", "wm_yr_wk"],
              how="left")

In [6]:
# Convert date
df["date"] = pd.to_datetime(df["date"])

# Sort for lag features
df = df.sort_values(["id", "date"])

# Lag features
for lag in [1, 7, 14, 28]:
    df[f"lag_{lag}"] = df.groupby("id")["sales"].shift(lag)

# Rolling means
df["rmean_7"] = df.groupby("id")["sales"].shift(1).rolling(7).mean()
df["rmean_28"] = df.groupby("id")["sales"].shift(1).rolling(28).mean()

# Date features
df["weekday"] = df["date"].dt.weekday
df["month"] = df["date"].dt.month

In [7]:
df

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,snap_CA,snap_TX,snap_WI,sell_price,lag_1,lag_7,lag_14,lag_28,rmean_7,rmean_28
1612,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1,3,2011-01-29,11101,...,0,0,0,2.00,NaN,NaN,NaN,NaN,NaN,NaN
32102,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,...,0,0,0,2.00,3.0,NaN,NaN,NaN,NaN,NaN
62592,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,...,0,0,0,2.00,0.0,NaN,NaN,NaN,NaN,NaN
93082,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_4,1,2011-02-01,11101,...,1,1,0,2.00,0.0,NaN,NaN,NaN,NaN,NaN
123572,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_5,4,2011-02-02,11101,...,1,0,1,2.00,1.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58203972,HOUSEHOLD_2_516_WI_3_validation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1909,0,2016-04-20,11612,...,0,0,0,5.94,0.0,0.0,0.0,0.0,0.0,0.0
58234462,HOUSEHOLD_2_516_WI_3_validation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1910,0,2016-04-21,11612,...,0,0,0,5.94,0.0,0.0,0.0,0.0,0.0,0.0
58264952,HOUSEHOLD_2_516_WI_3_validation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1911,0,2016-04-22,11612,...,0,0,0,5.94,0.0,0.0,0.0,0.0,0.0,0.0
58295442,HOUSEHOLD_2_516_WI_3_validation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,WI_3,WI,d_1912,0,2016-04-23,11613,...,0,0,0,5.94,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
df.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd',
       'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
       'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'lag_1', 'lag_7',
       'lag_14', 'lag_28', 'rmean_7', 'rmean_28'],
      dtype='object')

In [9]:
df["snap"] = np.nan

df.loc[df["state_id"] == "CA", "snap"] = df.loc[df["state_id"] == "CA", "snap_CA"]
df.loc[df["state_id"] == "TX", "snap"] = df.loc[df["state_id"] == "TX", "snap_TX"]
df.loc[df["state_id"] == "WI", "snap"] = df.loc[df["state_id"] == "WI", "snap_WI"]

In [10]:
categorical_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

In [11]:
df[categorical_cols] = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit_transform(df[categorical_cols])

In [12]:
# Drop NA (due to lagging)
df = df.dropna()

In [13]:
features = [
    "lag_1", "lag_7", "lag_14", "lag_28",
    "rmean_7", "rmean_28",
    "sell_price",
    "weekday", "month",
    "snap",
    # "item_id", "dept_id", "cat_id", "store_id", "state_id"
]

X = df[features]
y = df["sales"]

In [14]:
df["date"].max()

Timestamp('2014-06-15 00:00:00')

In [15]:
split_date = "2014-06-01"

X_train = X[df["date"] < split_date]
y_train = y[df["date"] < split_date]

X_valid = X[df["date"] >= split_date]
y_valid = y[df["date"] >= split_date]

In [16]:
X_train

,lag_1,lag_7,lag_14,lag_28,rmean_7,rmean_28,sell_price,weekday,month,snap
2593262,1.0,2.0,2.0,2.0,0.714286,0.821429,2.00,6,4,0.0
25216842,0.0,0.0,0.0,0.0,0.428571,0.428571,2.24,6,5,1.0
35888342,3.0,3.0,0.0,1.0,2.142857,1.000000,2.24,6,4,0.0
2596311,3.0,1.0,0.0,2.0,1.142857,0.857143,2.00,6,4,0.0
25219891,1.0,0.0,2.0,1.0,0.571429,0.892857,2.24,6,5,1.0
...,...,...,...,...,...,...,...,...,...,...
25241233,0.0,0.0,0.0,1.0,0.000000,0.035714,5.94,6,5,1.0
35912733,0.0,0.0,0.0,0.0,0.000000,0.000000,5.94,6,4,0.0
2620702,0.0,0.0,0.0,1.0,0.142857,0.178571,5.94,6,4,0.0
25244282,0.0,0.0,0.0,0.0,0.285714,0.107143,5.94,6,5,1.0


In [17]:
model = HistGradientBoostingRegressor(
    random_state=42,
    # categorical_features=[
        # 0, 0, 0, 0, 0, 0, 0,
        # 1, 1,
        # 1,
        # 1, 1, 1, 1, 1
    # ]
)

model.fit(X_train, y_train)

,loss,'squared_error'
,quantile,None
,learning_rate,0.1
,max_iter,100
,max_leaf_nodes,31
,max_depth,None
,min_samples_leaf,20
,l2_regularization,0.0
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'


In [18]:
preds = model.predict(X_valid)
rmse = root_mean_squared_error(y_valid, preds)

print("Validation RMSE:", rmse)

Validation RMSE: 2.4734434219427897


In [19]:
quantiles = [0.1, 0.5, 0.9]  # you can expand to full Kaggle set
models = {}

for q in quantiles:
    print(f"Training quantile {q}")
    
    model = HistGradientBoostingRegressor(
        loss="quantile",
        quantile=q,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    models[q] = model

Training quantile 0.1
Training quantile 0.5
Training quantile 0.9


In [20]:
predictions = {}

for q, model in models.items():
    predictions[q] = model.predict(X_train.tail(1000))  # example

# Convert to DataFrame
pred_df = pd.DataFrame(predictions)
print(pred_df.head())

   0.1     0.5       0.9
0  0.0  0.0000  1.126262
1  0.0  0.0000  1.852026
2  0.0  0.0000  0.982692
3  0.0  0.9997  2.839326
4  0.0  0.0000  0.879578
